# Capstone: Whose Reward Did We Teach?

Capstone notebook for the [Alignment & Safety](https://josephausterweil.github.io/probintro/ethics/alignment-safety/) chapter (Part IX) — the **last** chapter of the book.

**The experiment.** Chibany is auditing the robot mascot trainee. The trainee was tuned by **RLHF** — students compared pairs of its behaviors and said which was the better mascot — and Chibany wants to know what those preferences *actually* taught it.

You will:
- **Part A** — fit a **Bradley–Terry** reward model to the pairwise preferences and *recover the latent reward*, and see that it is identifiable only up to an additive constant.
- **Part B** — demonstrate **reward hacking**: optimize a policy against the *learned* reward and watch it exploit a gap the *true* reward would not — Goodhart's law in miniature.
- **Part C (stretch)** — show that adding *more diverse* preferences shrinks the ill-posedness, so the hack disappears.

The setup and data cells are complete — run them. The three parts are scaffolded **TODO**s (their driver lines are commented out, so *Run All* works top to bottom before you start). A reference solution is kept separate, mirroring the class `solutions/` pattern.

## 0. Setup (complete — run me)

In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
from jax import grad

print('JAX ready:', jax.__version__)

## 1. Chibany's audit data — behaviors and preferences (complete — run me)

Four things the trainee can do when a student arrives, each with a hidden 'good-mascot' value `TRUE_R` (what students actually care about — you never observe it directly). The data is **pairwise preferences**: for every pair of behaviors, `N_PER_PAIR` students compared them, and `COUNT_I` of them preferred the first. We generate the counts *deterministically* from Bradley–Terry, so the whole notebook is reproducible.

Parts B–C also need a **feature view** of each behavior: `HELP` (how much it helps a student — what we *mean* to reward) and `FLASH` (flashy showmanship — a proxy raters happen to like). On the 'normal' behaviors (0, 1, 2) the two features rise together; behavior 3 is the off-distribution one where they diverge.

In [ ]:
BEHAVIORS = ["ignore student", "wave", "hand over bento", "hand bento + bow"]
TRUE_R    = jnp.array([-1.5, -0.3, 1.0, 1.8])      # value of each behavior (HIDDEN from us)
K = len(BEHAVIORS)

# Pairwise preference counts, straight from Bradley-Terry (deterministic, no sampling).
PAIRS = jnp.array([[i, j] for i in range(K) for j in range(i + 1, K)])
N_PER_PAIR = 40
p_i     = jax.nn.sigmoid(TRUE_R[PAIRS[:, 0]] - TRUE_R[PAIRS[:, 1]])
COUNT_I = jnp.round(N_PER_PAIR * p_i)              # how many of the N preferred behavior i

# Feature view for Parts B-C: HELP == what we mean; FLASH == a proxy raters liked.
HELP  = jnp.array([1.0, 2.0, 3.0, 0.0])
FLASH = jnp.array([1.0, 2.0, 3.0, 9.0])           # equals HELP on the 'normal' behaviors 0,1,2

print(f"{int(PAIRS.shape[0] * N_PER_PAIR)} pairwise preferences over {K} behaviors, ready.")
print("pairs (i, j) and count preferring i:")
for (i, j), c in zip(PAIRS.tolist(), COUNT_I.tolist()):
    print(f"  {BEHAVIORS[i]:<18} vs {BEHAVIORS[j]:<18}  {int(c):>2}/{N_PER_PAIR} preferred the first")

## 2. Part A — fit a Bradley–Terry reward model — TODO

Recover the latent reward `r` from the preference counts by **maximum likelihood**. The negative log-likelihood is provided (it is exactly the chapter's); your job is the gradient-descent loop that minimizes it — the same `jax.grad` + loop pattern as [From Rules to Weights](https://josephausterweil.github.io/probintro/deep/from-rules-to-weights/).

Then check two things: the recovered reward (mean-centered) matches `TRUE_R`'s ranking and spacing, and starting the fit from a *shifted* initial guess (`jnp.full(K, 5.0)`) gives the **same differences** but a different absolute level — the additive-constant ambiguity.

In [ ]:
def bt_neg_loglik(r, pairs, count_i, n_per_pair):
    """Bradley-Terry negative log-likelihood of the counts under item rewards r."""
    gap = r[pairs[:, 0]] - r[pairs[:, 1]]
    ll  = count_i * jax.nn.log_sigmoid(gap) + (n_per_pair - count_i) * jax.nn.log_sigmoid(-gap)
    return -jnp.sum(ll)

def fit_reward_model(pairs, count_i, n_per_pair, K, r0=None, lr=0.02, steps=4000):
    """Recover the latent reward by gradient descent on the Bradley-Terry NLL.

    TODO:
      1. g = grad(bt_neg_loglik)          # gradient w.r.t. the FIRST arg, r
      2. r = jnp.zeros(K) if r0 is None else r0
      3. loop `steps` times:  r = r - lr * g(r, pairs, count_i, n_per_pair)
      4. return r
    """
    raise NotImplementedError

center = lambda v: v - jnp.mean(v)          # strip the unidentifiable additive constant

# --- uncomment once fit_reward_model is implemented ---
# r_hat = fit_reward_model(PAIRS, COUNT_I, N_PER_PAIR, K)
# r_shifted = fit_reward_model(PAIRS, COUNT_I, N_PER_PAIR, K, r0=jnp.full(K, 5.0))
# for k in range(K):
#     print(f"{BEHAVIORS[k]:<18} true {float(center(TRUE_R)[k]):+5.2f}   recovered {float(center(r_hat)[k]):+5.2f}")
# print("raw fit from 0 :", [round(float(x), 2) for x in r_hat])
# print("raw fit from +5:", [round(float(x), 2) for x in r_shifted])   # same differences

## 3. Part B — demonstrate reward hacking — TODO

The reward model is only an *approximation* of what we value, and the policy optimizes it *hard*. Here the preferences were all collected among 'normal' behaviors, where `HELP` and `FLASH` are perfectly correlated — so the fit **cannot tell them apart** (the ill-posedness of Part A, sharpened). A reward that credited `FLASH` explains every observed comparison exactly as well as one that credited `HELP`.

Show the consequence: take the flash-crediting reward and let the policy `argmax` it over the *full* menu, which includes the off-distribution behavior 3 ('flashy, unhelpful'). It picks the hack — high proxy, low true value — while the true reward never would.

In [ ]:
def learned_reward(w_help, w_flash):
    """A reward model linear in the two features."""
    return w_help * HELP + w_flash * FLASH

def show_reward_hacking():
    """Optimizing a reward that credited the PROXY hacks the true objective.

    TODO:
      1. On the SEEN behaviors (0,1,2), HELP == FLASH, so a help-crediting reward
         learned_reward(1, 0) and a flash-crediting one learned_reward(0, 1) rank
         them IDENTICALLY -- verify with jnp.argsort on each restricted to [0,1,2].
      2. argmax the flash-crediting reward over ALL FOUR behaviors.
      3. Compare with argmax of the TRUE value (HELP). The flash reward should pick
         behavior 3 ('flashy, unhelpful') -- the reward hack.
    """
    raise NotImplementedError

# show_reward_hacking()

## 4. Part C (stretch) — diversity shrinks the ill-posedness — TODO

The hack was possible only because the preference data never separated `HELP` from `FLASH`. Fix the *data*, not the optimizer: add comparisons that **break the correlation** — a helpful-but-plain behavior that students prefer over the flashy-but-unhelpful one. With those, the only reward that fits credits `HELP`, the recovered weight on `FLASH` collapses toward zero, and the `argmax` stops choosing the hack.

This is the honest lever for real alignment: you cannot rewrite human values, but you *can* collect preferences diverse enough to pin the reward down.

In [ ]:
def diverse_fit_kills_the_hack():
    """Add decorrelating preferences, refit, and show the hack disappears.

    TODO:
      1. Add a behavior with HIGH help, LOW flash (e.g. HELP=3, FLASH=0), and
         preferences in which students pick it OVER the flashy-unhelpful behavior.
      2. Refit a reward that is linear in (HELP, FLASH): fit weights (w_help, w_flash)
         by gradient descent on the Bradley-Terry likelihood of the AUGMENTED prefs
         (reuse bt_neg_loglik, but parameterize r = w_help*HELP + w_flash*FLASH).
      3. Show w_flash -> ~0 and that argmax of the refit reward no longer picks the
         flashy-unhelpful behavior. Diversity shrank the ill-posed set of fits.
    """
    raise NotImplementedError

# diverse_fit_kills_the_hack()

---

*This project is generously funded by the Japanese Probabilistic Computing Consortium Association ([JPCCA](https://jpcca.org/)).*